# Red Wine Quality Analysis

**Exploring physicochemical properties of red wine to understand what makes wine quality higher or lower.**

## Project Overview

This project analyzes the Red Wine Quality dataset to understand how chemical properties like acidity, sugar, sulfur dioxide, alcohol content, and pH relate to wine quality ratings. The dataset contains expert quality ratings from 0-10.

## Learning Objectives

1. Analyze the relationship between chemical properties and a quality score
2. Handle ordinal target variables in EDA
3. Identify the strongest predictors of quality through correlation and visualization
4. Apply statistical tests across quality groups
5. Understand class imbalance in rating data

## Business / Research Problem

Wine quality assessment is traditionally done by expert tasters — an expensive and subjective process. Understanding chemical correlates of quality could:
- **Automate quality control** in wine production
- **Guide winemakers** on which chemical properties to optimize
- **Reduce costs** of quality certification

**Key question:** *Which chemical properties best predict wine quality, and how do they interact?*

## Why This Analysis Matters

The wine industry is worth billions globally. Objective, data-driven quality assessment can improve consistency, reduce costs, and help producers target specific quality tiers.

## Dataset Overview

| Feature | Description |
|---------|-------------|
| `fixed acidity` | Tartaric acid (g/dm³) |
| `volatile acidity` | Acetic acid (g/dm³) — too high = vinegar taste |
| `citric acid` | Freshness/flavor (g/dm³) |
| `residual sugar` | Sugar remaining after fermentation |
| `chlorides` | Salt content |
| `free sulfur dioxide` | Free SO₂ — prevents microbial growth |
| `total sulfur dioxide` | Total SO₂ |
| `density` | Density of wine |
| `pH` | Acidity level (3-4 range) |
| `sulphates` | SO₂ additive |
| `alcohol` | Alcohol percentage |
| `quality` | Score 0-10 (target) |

**Size:** ~1,599 samples

## Dataset Source and License Notes- **Source:** [Kaggle - Red Wine Quality](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009)- **Original:** UCI ML Repository (Cortez et al., 2009)- **License:** CC BY 4.0

## Environment Setup

In [1]:
!pip install pandas numpy matplotlib seaborn scipy kagglehub -q

## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded successfully.')

All imports loaded successfully.


## Configuration / Constants

In [3]:
RANDOM_SEED = 42
KAGGLE_DATASET = 'uciml/red-wine-quality-cortez-et-al-2009'
SIGNIFICANCE_LEVEL = 0.05
np.random.seed(RANDOM_SEED)

## Dataset Download / Loading

In [4]:
import kagglehub
import os

path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f'Downloaded to: {path}')

csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
print(f'Loaded {len(df)} rows and {len(df.columns)} columns')
df.head()

Downloaded to: C:\Users\ahmad\.cache\kagglehub\datasets\uciml\red-wine-quality-cortez-et-al-2009\versions\2
Loaded 1599 rows and 12 columns


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


## Data Validation Checks

In [5]:
print(f'Shape: {df.shape}')
print(f'\nMissing: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'\nQuality distribution:')
print(df['quality'].value_counts().sort_index())
df.describe()

Shape: (1599, 12)

Missing: 0
Duplicates: 240

Quality distribution:
quality
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


## Data Cleaning

In [6]:
df = df.drop_duplicates()
print(f'After dedup: {len(df)} rows')

# Create quality categories
df['quality_cat'] = pd.cut(df['quality'], bins=[0, 4, 6, 10],
                            labels=['Low', 'Medium', 'High'])
print(f'\nQuality categories:\n{df["quality_cat"].value_counts()}')
print('\nNote: Most wines are in the Medium (5-6) range — class imbalance.')

After dedup: 1359 rows

Quality categories:
quality_cat
Medium    1112
High       184
Low         63
Name: count, dtype: int64

Note: Most wines are in the Medium (5-6) range — class imbalance.


## Exploratory Data Analysis

In [7]:
print('=== Feature means by quality ===')
print(df.groupby('quality').mean(numeric_only=True).round(3))

=== Feature means by quality ===
         fixed acidity  volatile acidity  citric acid  residual sugar  \
quality                                                                 
3                8.360             0.885        0.171           2.635   
4                7.779             0.694        0.174           2.694   
5                8.171             0.579        0.245           2.510   
6                8.337             0.496        0.279           2.457   
7                8.859             0.404        0.372           2.717   
8                8.441             0.428        0.383           2.576   

         chlorides  free sulfur dioxide  total sulfur dioxide  density     pH  \
quality                                                                         
3            0.122               11.000                24.900    0.997  3.398   
4            0.091               12.264                36.245    0.997  3.382   
5            0.094               17.161                57.

## Univariate Analysis

In [8]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
features = [c for c in df.columns if c not in ['quality', 'quality_cat']]
for i, col in enumerate(features):
    ax = axes[i // 4, i % 4]
    ax.hist(df[col], bins=25, color='steelblue', edgecolor='white')
    ax.set_title(col, fontsize=10)
# Hide empty subplot if any
for j in range(len(features), 12):
    axes[j // 4, j % 4].set_visible(False)
plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

In [9]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
key_feats = ['alcohol', 'volatile acidity', 'sulphates', 'citric acid', 'total sulfur dioxide', 'pH']
for i, feat in enumerate(key_feats):
    ax = axes[i // 3, i % 3]
    sns.boxplot(data=df, x='quality', y=feat, ax=ax, palette='viridis')
    ax.set_title(f'{feat} by Quality')
plt.tight_layout()
plt.show()

## Feature-Specific Insights

In [10]:
# Correlation with quality
corr_quality = df.drop(columns=['quality_cat']).corr()['quality'].drop('quality').sort_values(key=abs, ascending=False)
print('=== Feature Correlation with Quality ===')
print(corr_quality.round(4))

fig, ax = plt.subplots(figsize=(10, 6))
corr_quality.plot(kind='barh', ax=ax, color=['coral' if v < 0 else 'steelblue' for v in corr_quality])
ax.set_title('Feature Correlation with Quality')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

=== Feature Correlation with Quality ===
alcohol                 0.4803
volatile acidity       -0.3952
sulphates               0.2488
citric acid             0.2281
density                -0.1843
total sulfur dioxide   -0.1779
chlorides              -0.1310
fixed acidity           0.1190
pH                     -0.0552
free sulfur dioxide    -0.0505
residual sugar          0.0136
Name: quality, dtype: float64


## Statistical Checks / Hypothesis Testing

In [11]:
# ANOVA for key features across quality levels
print('=== ANOVA Tests ===')
for feat in ['alcohol', 'volatile acidity', 'sulphates', 'citric acid']:
    groups = [g[feat].values for _, g in df.groupby('quality')]
    f_stat, p_val = stats.f_oneway(*groups)
    sig = 'Significant' if p_val < SIGNIFICANCE_LEVEL else 'Not significant'
    print(f'{feat}: F={f_stat:.2f}, p={p_val:.2e} ({sig})')

=== ANOVA Tests ===
alcohol: F=103.44, p=1.52e-92 (Significant)
volatile acidity: F=53.44, p=9.99e-51 (Significant)
sulphates: F=18.54, p=7.59e-18 (Significant)
citric acid: F=16.29, p=1.24e-15 (Significant)


In [12]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.drop(columns=['quality_cat']).corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Visual Analysis

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(df['alcohol'], df['volatile acidity'], c=df['quality'],
                cmap='viridis', alpha=0.5, s=20)
axes[0].set_xlabel('Alcohol')
axes[0].set_ylabel('Volatile Acidity')
axes[0].set_title('Alcohol vs Volatile Acidity (colored by quality)')

df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[1],
    color='steelblue', edgecolor='white')
axes[1].set_title('Quality Score Distribution')
axes[1].set_xlabel('Quality')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## Key Findings

1. **Alcohol is the strongest positive predictor** of quality
2. **Volatile acidity is the strongest negative predictor** — higher = worse quality
3. **Sulphates and citric acid** positively correlate with quality
4. **Most wines score 5-6** — extreme quality ratings are rare
5. **pH has weak correlation** with quality despite being important in winemaking
6. **Fixed acidity and residual sugar** show minimal quality correlation

## Limitations

1. **Red wine only:** Results don't apply to white wine
2. **Class imbalance:** Very few wines rated 3-4 or 8-9
3. **Expert subjectivity:** Quality scores are subjective human ratings
4. **Limited features:** Grape variety, vintage, region are not included
5. **Small dataset:** ~1,600 samples limits complex modeling

## Recommendations / Next Steps

1. Build a quality prediction model (Random Forest works well here)
2. Treat as binary classification (good/bad) to handle class imbalance
3. Compare with white wine dataset

### How to Extend This Analysis
- Apply SMOTE for class balancing
- Build an interactive wine quality predictor
- Analyze feature interactions (e.g., alcohol × sulphates)

## Common Mistakes

1. **Treating quality as continuous:** It's ordinal — use appropriate metrics
2. **Ignoring class imbalance:** Accuracy is misleading when 80% of wines score 5-6
3. **Assuming correlation = causation:** High alcohol doesn't cause quality — winemaking is complex
4. **Not scaling features:** Chemical properties have very different ranges
5. **Removing 'outlier' wines:** Extreme quality scores are rare but real

## Mini Challenge / Exercises

1. Create a binary target: 'good' (quality >= 7) vs 'not good'. What percentage is 'good'?
2. Which single feature gives the best separation between good and not-good wines?
3. Are sulphates and alcohol correlated with each other, or independently predictive?
4. Create a violin plot of alcohol by quality_cat (Low/Medium/High)
5. Build a simple logistic regression to predict good vs not-good

In [14]:
# Space for exercise solutions

## Final Summary / Key Takeaways

- **Alcohol and volatile acidity** are the two most important quality indicators
- **Higher alcohol, lower volatile acidity, more sulphates** = higher quality
- **Quality scores cluster around 5-6** — true outlier wines are rare
- **Chemical properties alone can partially predict quality** but miss subjective elements
- **This dataset is excellent for practicing** classification, imbalanced learning, and feature importance

The wine quality dataset beautifully demonstrates the challenges of predicting subjective outcomes from objective measurements.